# Exploratory Data Analysis — Sales Analytics

Run `python src/generate_synthetic_data.py` first if you don't have `data/sales.csv`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

df = pd.read_csv('../data/sales.csv', parse_dates=['OrderDate'])
df.shape

In [ ]:
df.head()

## Revenue over time

In [ ]:
monthly = df.set_index('OrderDate').resample('MS')['Sales'].sum()

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, marker='o', markersize=3)
ax.set_title('Monthly Revenue')
ax.set_ylabel('Revenue ($)')
plt.show()

## Category & region mix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cat_revenue = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
sns.barplot(x=cat_revenue.values, y=cat_revenue.index, ax=axes[0])
axes[0].set_title('Revenue by Category')
axes[0].set_xlabel('Revenue ($)')

region_revenue = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
sns.barplot(x=region_revenue.values, y=region_revenue.index, ax=axes[1])
axes[1].set_title('Revenue by Region')
axes[1].set_xlabel('Revenue ($)')

plt.tight_layout()
plt.show()

## Order value distribution

In [ ]:
order_totals = df.groupby('OrderID')['Sales'].sum()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(order_totals, bins=50, ax=ax)
ax.axvline(order_totals.mean(), color='crimson', linestyle='--', label=f'Mean: ${order_totals.mean():.0f}')
ax.axvline(order_totals.median(), color='darkorange', linestyle='--', label=f'Median: ${order_totals.median():.0f}')
ax.set_title('Order Value Distribution')
ax.set_xlabel('Order total ($)')
ax.legend()
plt.show()

## Customer purchase frequency

A quick look at how concentrated revenue is among repeat customers -- this motivates the RFM segmentation in `src/rfm_segmentation.py`.

In [ ]:
customer_orders = df.groupby('CustomerID')['OrderID'].nunique()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(customer_orders, bins=30, ax=ax)
ax.set_title('Orders per Customer')
ax.set_xlabel('Number of orders')
plt.show()

one_time = (customer_orders == 1).mean() * 100
print(f"{one_time:.1f}% of customers have placed exactly one order.")

## Takeaways

- Revenue shows clear seasonality (see `src/kpi_analysis.py` for the full monthly trend).
- A meaningful share of customers are one-time buyers -- retention is a real lever here, not a rounding error. See `src/cohort_analysis.py`.
- Order values are right-skewed, so median (not mean) is the more representative "typical order" figure.

Next: `src/rfm_segmentation.py` for customer segments, `src/cohort_analysis.py` for retention, `src/forecasting.py` for the revenue forecast.